# 2 · Building and running a graph

Each node in the graph binds its inputs: `From` an output, a `Static` value, or nothing (the default applies). `CompiledGraph.from_graph` checks the graph and never raises for a fault in it. `compiled.stream()` runs it as a stream of events; in a notebook, `await compiled.run()` returns the state the run ended on.

In [ ]:
from typing import Annotated

from conductor import node
from conductor import CompiledGraph, From, Graph, GraphNode, NodeRegistry, Param, Ref, Result, Series, Static, node
from conductor.widgets import TextWidget
from conductor_nodes.types import Text


@node("echo", title="Echo", description="Returns its input", category="text")
def echo(text: Annotated[Text, Param(title="Input", widget=TextWidget())]) -> Annotated[Text, Result(title="Output")]:
    return text


@node("upper", title="Uppercase", description="Uppercases a text", category="text")
def upper(text: Annotated[Text, Param(title="Input", widget=TextWidget())]) -> Annotated[Text, Result(title="Output")]:
    return Text(text.upper())


@node("pair", title="Pair", description="Joins two texts", category="text")
def pair(
    a: Annotated[Text, Param(title="A", widget=TextWidget())],
    b: Annotated[Text, Param(title="B", widget=TextWidget())],
    separator: Annotated[Text, Param(title="Separator", widget=TextWidget())] = Text(" + "),
) -> Annotated[Text, Result(title="Result")]:
    return Text(f"{a}{separator}{b}")


registry = NodeRegistry()
for cls in (echo, upper, pair):
    registry = registry.extended_with([cls])

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

In [ ]:
graph = Graph(nodes=[
    GraphNode(id="hello", type="echo", version=1, bindings={"text": Static("hello")}),
    GraphNode(id="world", type="echo", version=1, bindings={"text": Static("world")}),
    GraphNode(id="shout", type="upper", version=1, bindings={"text": From("hello.result")}),
    GraphNode(id="pair", type="pair", version=1, bindings={
        "a": From("shout.result"),
        "b": From("world.result"),
        "separator": Static(" & "),
    }),
])

compiled = CompiledGraph.from_graph(graph, registry)
print("runnable:", compiled.is_runnable)
print("order:   ", compiled.execution_order)

## Ask the compiled graph

`compiled.node(node_id)` for a node, `compiled.field(ref)` for a field.

In [ ]:
print("pair.a comes from:", compiled.field(Ref("pair", "a")).binding)
print("pair.a receives:  ", compiled.run_plan.nodes["pair"].inputs["a"].arrives)
print("pair.a is:        ", compiled.field(Ref("pair", "a")).type.describe())

## Draw it

`flowchart(compiled)` is Mermaid source, each edge labelled with how its input receives it.

In [ ]:
from conductor_providers.mermaid import flowchart

print(flowchart(compiled))

## Problems, not exceptions

A broken graph compiles. Each problem has a stable `code`; `execute` refuses a fatal one.

In [ ]:
broken = CompiledGraph.from_graph(
    Graph(nodes=[GraphNode(id="shout", type="upper", version=1, bindings={"text": From("ghost.result")})]),
    registry,
)
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.type} on {problem.loc[0]}: {problem.msg}")

## Run it

A run is its `RunState`; `compiled.results(state)` is `{node_id: {output: value}}`.

In [ ]:
state = await compiled.run()
results = compiled.results(state)
for node_id, outputs in results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

`with_inputs` fills inputs by bare name, or by address when two share a name. `state.outputs` reads what the graph returns, by address.

In [ ]:
print("takes:  ", [str(i.name) for i in compiled.interface.inputs])
print("returns:", [str(o.name) for o in compiled.interface.outputs])

ready = compiled.with_inputs(**{"hello.text": "goodbye"})
state = await ready.run()
ready.outputs(state)

## Watch the events

In [ ]:
async with compiled.stream() as events:
  async for event in events:
    match event.type:
        case "node_start":
            print(f"  start     {event.node_id}")
        case "node_complete":
            print(f"  complete  {event.node_id}: {event.result['result']!r}")
        case "run_end":
            print(f"  done      {sorted(compiled.results(event.state))}")
        case other:
            print(f"  {other}")

## Once per row

A list on an input declared for one value runs the node once per value. A `Series[X]` input takes the whole series in one call.

In [ ]:
from conductor import node
@node("join-all", title="Join all", description="Joins every text of a series", category="text")
def join_all(texts: Annotated[Series[Text], Param(title="Texts")]) -> Annotated[Text, Result(title="Result")]:
    return Text(" ".join(texts))


registry = registry.extended_with([join_all])

rows = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="shout", type="upper", version=1, bindings={"text": Static(["one", "two", "three"])}),
        GraphNode(id="all", type="join-all", version=1, bindings={"texts": From("shout.result")}),
    ]),
    registry,
)
print(rows.run_plan)

async with rows.stream() as events:
    async for event in events:
        if event.type == "node_progress":
            print(f"  {event.node_id}: {event.done} of {event.total}")
        elif event.type == "run_end":
            results = rows.results(event.state)
            print("  shout:", list(results["shout"]["result"]))
            print("  all:  ", results["all"]["result"])

## Save it

A `Graph` reads back what it wrote, as YAML or JSON.

In [ ]:
text = graph.to_yaml()
print(text)
assert Graph.from_yaml(text) == graph